Transformers: Transformers are a deep learning architecture with three main layers: embeddings, attention, and feedforward neural networks (FNN).

Large Language Models(LLMs): These are deep learning models that have been pretrained on a massive amount of text data.

The transformer architecture: It refers to the series of layers and computations that the input data passes through to produce a final result.

# Basic Sentiment Analysis

### Sentiment Analysis is used to determine the positivity or negativity of the text.

In [26]:
import pandas as pd
import warnings

pd.set_option('display.max_colwidth',None)
warnings.filterwarnings('ignore')

data=pd.read_excel(r'C:\Users\HP\Desktop\PG\NLP\NLP with ML\data\Popchip_Reviews.xlsx')
data.sample(5)

,Id,UserId,Rating,Priority,Title,Text
453,24142,AOZ8OAFA0MF9N,1,High,"Other Flavors are Great, these Not So Much","The name of this flavor sounds fantastic, and I wondered why they didn't include them in their multi-flavor assortment. Then I tasted them. These chips are terrible, there is no parmesan flavor to speak of, and they somehow messed up the garlic as well. Avoid these and the also-no-incuded-assortment flavor of ""jalapeno"". Very disappointed.<br /><br />The cheddar, vinegar & salt and peppered flavors are FANTASTIC!"
142,23831,A1WBEIH7Z37LN9,5,Low,Surprisingly delicious!,"Wow, these are really good! At first, I didn't understand ""Popchips"", and I was actually expecting something like regular fried potato chips, only made from sweet potatos. The Popchips are very different from that!<br /><br />These contain rice flour as well as sweet potatos, and they are kind of air-popped, very light like popcorn. Texture-wise, they remind me of ""Munchos"" only lighter and without the oily feel and taste. Remember Munchos? These are WAY better.<br /><br />Next thing - the flavor from the sweet potatos is YUMMY! I know when I first read ""rice flour"" in the ingredients, I was thinking those cardboard-like rice cakes. The sweet potato completely takes away any blandness or boringness and adds good taste. And there is just a touch of salt - just enough to bring out the YUM without overdoing it.<br /><br />Finally, the package size. When I saw ""0.8 ounce"", I expected that I would want two, three, or even four bags of these at a time. Nope - one bag is a good snack! Not that 0.8 is a big bag, or ""filling"" or anything like that, but the bag is just the right size, which is a very pleasant surpise to me! I finish the last chips, like the last of the salt from my fingers, and I feel satisfied - like, wow, that was tasty and just right! At Amazon's prices, I believe I will be buying these regularly!"
431,24120,A3GGQC9BRC6ABK,5,Low,Amazing Chips!,"These chips are really good. Crisp and delicious. My favorite flavor is the salt and pepper, my second in the garlic parm, but they were all good. (I'm not even a fan of BBQ anything, but the BBQ chips were alright).<br /><br />Only 110 calories for a small bag. Not to bad if i say so myself."
80,23769,A22IEMCYCS128S,5,Low,Very good,"These chips are tasty and meet the need I sometimes have for a ""treat"". Usually potato chips are off limits due to fat and salt.<br /><br />These fill the need for that ""treat"". The individual packages help limit the amount I eat and are very good."
440,24129,A35Q0RBM3YNQNF,5,Low,Healthy & Delicious Potato Chips,"Oh, these chips are wonderful. They are not greasy, have a very satisfying crunch and although they are formed chips, they taste like actual potato chips. Only one grocery store in my area carries these chips and they are frequently out of stock, even at $3 a bag. Worse, even when in stock, I am lucky to find more than a few bags at a time in the store. I hate making the special trip just for the chips, particularly when I often find the shelf empty. Problem solved -- Amazon's are much more reasonably priced and I can stock up -- and this snack is as close in flavor to actual potato chips that I've found to date."


### Using VADER (Traditional NLP)

In [27]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer=SentimentIntensityAnalyzer()
data['Vader Sentiment Scores']=data['Text'].apply(lambda x: analyzer.polarity_scores(x)['compound'])
data.head()

,Id,UserId,Rating,Priority,Title,Text,Vader Sentiment Scores
0,23689,A21SYVGVNG8RAS,5,Low,Yummy snacks!,Popchips are the bomb!! I use the parmesan garlic to scoop up cottage cheese as a healthy alternative to chips and dip. My healthy eating program is saved.,0.9244
1,23690,AQJYXC0MPRQJL,5,Low,Great chip that is different from the rest,"I like the puffed nature of this chip that makes it more unique in the chip market. I ordered the Salt and Vinegar and absolutely love that flavor, hands down my favorite chip ever. I have tried the cheddar and regular flavors as well. The cheddar is about a 4/5 and the regular is about a 3/5 because I prefer strong flavors and obviously that would not be the case for the regular. The Salt and Vinegar is kind of weak compared to some regular S&V chips, but is quite flavorful and makes you wanting to come back for more.",0.7269
2,23691,A30NYUHEDLWI0Y,5,Low,Great Alternative to Potato Chips,"I just love these chips! I was always a big fan of potato chips, but haven't had one since I discovered popchips. They are great for dipping or all alone. I am constantly re-ordering them. One note however-if you are on a low salt diet these chips are probably not for you. They are high in sodium. We go through a case every two months. If you love them it pays to join the subscribe and save program through Amazon. You save money and stay supplied!",0.9790
3,23692,A2NU55U9LKTB5J,3,High,Not somthing I would crave,"These tasted like potatoe stix, that we got in grade school with our lunches usually on pizza day. They were the bomb then, not so much now. Won't buy again unless I get them for cheap or free.",0.8689
4,23693,A225F7QFP5LIW2,5,Low,healthy and delicious,"These chips are great! They look almost like a flattened rice cake, but taste so much better, more like a potato chip. The bbq flavor is delicious. They are very low in fat and full of flavor. It is easy to eat an entire bag of these!",0.9613


### Using Transformer Pipeline(Modern NLP)

In [28]:
%%time

from transformers import pipeline
from transformers import logging

logging.set_verbosity_error()


CPU times: total: 125 ms
Wall time: 242 ms


In [29]:
%%time 

sentiment_analyzer=pipeline('sentiment-analysis',
                            model='distilbert/distilbert-base-uncased-finetuned-sst-2-english',
                            device=-1,
                            truncation=True
                           )

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 1051.12it/s]


CPU times: total: 656 ms
Wall time: 3.23 s


In [30]:
sentiment_analyzer(data['Text'][0])

[{'label': 'POSITIVE', 'score': 0.9935212731361389}]

In [31]:
%%time

data['Transformer Sentiment Scores']=data['Text'].apply(lambda x: sentiment_analyzer(x)[0]['score'])
data.head()
# data.Text.apply(sentiment_analyzer)

CPU times: total: 9min 15s
Wall time: 1min 37s


,Id,UserId,Rating,Priority,Title,Text,Vader Sentiment Scores,Transformer Sentiment Scores
0,23689,A21SYVGVNG8RAS,5,Low,Yummy snacks!,Popchips are the bomb!! I use the parmesan garlic to scoop up cottage cheese as a healthy alternative to chips and dip. My healthy eating program is saved.,0.9244,0.993521
1,23690,AQJYXC0MPRQJL,5,Low,Great chip that is different from the rest,"I like the puffed nature of this chip that makes it more unique in the chip market. I ordered the Salt and Vinegar and absolutely love that flavor, hands down my favorite chip ever. I have tried the cheddar and regular flavors as well. The cheddar is about a 4/5 and the regular is about a 3/5 because I prefer strong flavors and obviously that would not be the case for the regular. The Salt and Vinegar is kind of weak compared to some regular S&V chips, but is quite flavorful and makes you wanting to come back for more.",0.7269,0.999605
2,23691,A30NYUHEDLWI0Y,5,Low,Great Alternative to Potato Chips,"I just love these chips! I was always a big fan of potato chips, but haven't had one since I discovered popchips. They are great for dipping or all alone. I am constantly re-ordering them. One note however-if you are on a low salt diet these chips are probably not for you. They are high in sodium. We go through a case every two months. If you love them it pays to join the subscribe and save program through Amazon. You save money and stay supplied!",0.9790,0.698488
3,23692,A2NU55U9LKTB5J,3,High,Not somthing I would crave,"These tasted like potatoe stix, that we got in grade school with our lunches usually on pizza day. They were the bomb then, not so much now. Won't buy again unless I get them for cheap or free.",0.8689,0.999631
4,23693,A225F7QFP5LIW2,5,Low,healthy and delicious,"These chips are great! They look almost like a flattened rice cake, but taste so much better, more like a potato chip. The bbq flavor is delicious. They are very low in fat and full of flavor. It is easy to eat an entire bag of these!",0.9613,0.999181


In [32]:
data[['Text','Vader Sentiment Scores','Transformer Sentiment Scores']]

,Text,Vader Sentiment Scores,Transformer Sentiment Scores
0,Popchips are the bomb!! I use the parmesan garlic to scoop up cottage cheese as a healthy alternative to chips and dip. My healthy eating program is saved.,0.9244,0.993521
1,"I like the puffed nature of this chip that makes it more unique in the chip market. I ordered the Salt and Vinegar and absolutely love that flavor, hands down my favorite chip ever. I have tried the cheddar and regular flavors as well. The cheddar is about a 4/5 and the regular is about a 3/5 because I prefer strong flavors and obviously that would not be the case for the regular. The Salt and Vinegar is kind of weak compared to some regular S&V chips, but is quite flavorful and makes you wanting to come back for more.",0.7269,0.999605
2,"I just love these chips! I was always a big fan of potato chips, but haven't had one since I discovered popchips. They are great for dipping or all alone. I am constantly re-ordering them. One note however-if you are on a low salt diet these chips are probably not for you. They are high in sodium. We go through a case every two months. If you love them it pays to join the subscribe and save program through Amazon. You save money and stay supplied!",0.9790,0.698488
3,"These tasted like potatoe stix, that we got in grade school with our lunches usually on pizza day. They were the bomb then, not so much now. Won't buy again unless I get them for cheap or free.",0.8689,0.999631
4,"These chips are great! They look almost like a flattened rice cake, but taste so much better, more like a potato chip. The bbq flavor is delicious. They are very low in fat and full of flavor. It is easy to eat an entire bag of these!",0.9613,0.999181
...,...,...,...
559,"I love potato chips. I could eat them by the bagful but thanks to the powers that be, this would not provide anyone with enough nutrition to survive. Nonetheless I have eaten my share of potato chips over the years, and perhaps as a result I have been watching my weight lately. I discovered these great popchips and they hit the spot in a number of ways. First they are a low-calorie alternative to regular potato chips, meaning I can eat a normal-sized portion and still keep on target with my weight loss goals. Second, they are gluten-free, which is great for those of us who have gluten intolerance issues. Third, they taste great - light and airy, crispy, rich in that great potato flavor that keeps me reaching for another one. This is a great product and I hope it stays around for a good long time.",0.9951,0.996187
560,"When PopChips were really hard to find, I was ordering them by the case from Amazon on a regular basis. The price was always great and these really are my go-to snack. They are always fresh and delicious from Amazon.<br /><br />I definitely prefer Original over barbecue and sour cream & onion. I haven't tried other flavors, but Original works just fine for me because I use it for various dips and will even use dry seasonings on them if I'm in the mood.<br /><br />I don't know anybody that I've offered some of these to who didn't want a bag of their own! Highly recommended.",0.9300,0.999685
561,These are a much healthy alternative to most chips and they taste great. They have a great crunch and flavor and don't have that bad after taste that most baked chips have. My only regret is that I didn't order more when they were on sale there all gone now but even at regular price are worth it.,0.8087,0.999781
562,"These are so good, I've started getting them automatically. I like the original flavor, but they have a lot of others.",0.5818,0.999816
